In [1]:
import os
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam

print("TensorFlow version:", tf.__version__)
print("NumPy version:", np.__version__)

TensorFlow version: 2.16.1
NumPy version: 1.26.4


In [4]:
# ============================================================
# SETTINGS
# ============================================================

DATASET_DIR = r"D:\ISL_Recognition_Project\dataset"
MODEL_DIR = r"D:\ISL_Recognition_Project\models"

MODEL_PATH = os.path.join(MODEL_DIR, "sign_model.keras")
NORMALIZATION_PATH = os.path.join(MODEL_DIR, "normalization.npz")

FRAMES = 30
FEATURES = 255

ACTIONS = [
    "Hello",
    "Thank You",
    "Help",
    "Yes",
    "No",
    "Please",
    "Sorry",
    "Good Morning",
    "Good Night",
    "How Are You",
    "Welcome",
    "Come",
    "Go",
    "Stop",
    "Eat",
    "Drink",
    "Water",
    "Home",
    "Friend",
    "Love"
]

NUM_CLASSES = len(ACTIONS)

os.makedirs(MODEL_DIR, exist_ok=True)

print("Dataset folder:", DATASET_DIR)
print("Model folder:", MODEL_DIR)
print("Number of signs:", NUM_CLASSES)

Dataset folder: D:\ISL_Recognition_Project\dataset
Model folder: D:\ISL_Recognition_Project\models
Number of signs: 20


In [5]:
# ============================================================
# CHECK DATASET FOLDERS
# ============================================================

print("Checking dataset folders...\n")

for action in ACTIONS:
    action_path = os.path.join(DATASET_DIR, action)

    if os.path.isdir(action_path):
        print(f"✓ {action}")
    else:
        print(f"✗ MISSING: {action}")

Checking dataset folders...

✓ Hello
✓ Thank You
✓ Help
✓ Yes
✓ No
✓ Please
✓ Sorry
✓ Good Morning
✓ Good Night
✓ How Are You
✓ Welcome
✓ Come
✓ Go
✓ Stop
✓ Eat
✓ Drink
✓ Water
✓ Home
✓ Friend
✓ Love


In [6]:
# ============================================================
# LOAD DATASET
# ============================================================

X = []
y = []

print("Loading dataset...\n")

for label, action in enumerate(ACTIONS):

    action_path = os.path.join(DATASET_DIR, action)

    files = sorted([
        file
        for file in os.listdir(action_path)
        if file.lower().endswith(".npy")
    ])

    print(f"{action:15s}: {len(files)} sequences")

    if len(files) == 0:
        raise ValueError(
            f"No .npy files found for sign: {action}"
        )

    for file in files:

        file_path = os.path.join(action_path, file)

        sequence = np.load(file_path)

        # Check shape
        if sequence.shape != (FRAMES, FEATURES):
            raise ValueError(
                f"\nWrong shape in:\n{file_path}\n"
                f"Expected: {(FRAMES, FEATURES)}\n"
                f"Found:    {sequence.shape}"
            )

        # Check NaN / infinity
        if not np.isfinite(sequence).all():
            raise ValueError(
                f"\nNaN or infinity found in:\n{file_path}"
            )

        X.append(sequence.astype(np.float32))
        y.append(label)


X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int32)

print("\n========================================")
print("DATASET LOADED")
print("========================================")

print("Total sequences :", len(X))
print("Input shape     :", X.shape)
print("Labels shape    :", y.shape)
print("Number of signs :", NUM_CLASSES)

Loading dataset...

Hello          : 10 sequences
Thank You      : 10 sequences
Help           : 10 sequences
Yes            : 10 sequences
No             : 10 sequences
Please         : 10 sequences
Sorry          : 10 sequences
Good Morning   : 10 sequences
Good Night     : 10 sequences
How Are You    : 10 sequences
Welcome        : 10 sequences
Come           : 10 sequences
Go             : 10 sequences
Stop           : 10 sequences
Eat            : 10 sequences
Drink          : 10 sequences
Water          : 10 sequences
Home           : 10 sequences
Friend         : 10 sequences
Love           : 10 sequences

DATASET LOADED
Total sequences : 200
Input shape     : (200, 30, 255)
Labels shape    : (200,)
Number of signs : 20


In [7]:
# ============================================================
# DATASET COUNT
# ============================================================

print("\nSequences per sign:\n")

for label, action in enumerate(ACTIONS):
    count = np.sum(y == label)
    print(f"{action:15s}: {count}")


Sequences per sign:

Hello          : 10
Thank You      : 10
Help           : 10
Yes            : 10
No             : 10
Please         : 10
Sorry          : 10
Good Morning   : 10
Good Night     : 10
How Are You    : 10
Welcome        : 10
Come           : 10
Go             : 10
Stop           : 10
Eat            : 10
Drink          : 10
Water          : 10
Home           : 10
Friend         : 10
Love           : 10


In [8]:
# ============================================================
# TRAIN / VALIDATION SPLIT
# ============================================================

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("========================================")
print("DATA SPLIT")
print("========================================")

print("Training samples   :", len(X_train))
print("Validation samples :", len(X_val))

print("\nTraining shape   :", X_train.shape)
print("Validation shape :", X_val.shape)

DATA SPLIT
Training samples   : 160
Validation samples : 40

Training shape   : (160, 30, 255)
Validation shape : (40, 30, 255)


In [10]:
# ============================================================
# NORMALIZATION
# ============================================================

print("Calculating normalization values...")

mean = X_train.mean(
    axis=(0, 1),
    keepdims=True
)

std = X_train.std(
    axis=(0, 1),
    keepdims=True
)

# Prevent division by zero
std = np.where(std < 1e-6, 1.0, std)

# Normalize training and validation data
X_train = (X_train - mean) / std
X_val = (X_val - mean) / std

X_train = X_train.astype(np.float32)
X_val = X_val.astype(np.float32)

print("Normalization completed.")

Calculating normalization values...
Normalization completed.


In [11]:
# ============================================================
# SAVE NORMALIZATION VALUES
# ============================================================

np.savez(
    NORMALIZATION_PATH,
    mean=mean.astype(np.float32),
    std=std.astype(np.float32)
)

print("Normalization saved to:")
print(NORMALIZATION_PATH)

Normalization saved to:
D:\ISL_Recognition_Project\models\normalization.npz


In [12]:
# ============================================================
# ONE-HOT ENCODING
# ============================================================

y_train_cat = to_categorical(
    y_train,
    num_classes=NUM_CLASSES
)

y_val_cat = to_categorical(
    y_val,
    num_classes=NUM_CLASSES
)

print("Training labels:", y_train_cat.shape)
print("Validation labels:", y_val_cat.shape)

Training labels: (160, 20)
Validation labels: (40, 20)


In [13]:
# ============================================================
# BUILD LSTM MODEL
# ============================================================

model = Sequential([

    LSTM(
        128,
        return_sequences=True,
        activation="tanh",
        input_shape=(FRAMES, FEATURES)
    ),

    BatchNormalization(),
    Dropout(0.30),

    LSTM(
        128,
        return_sequences=True,
        activation="tanh"
    ),

    BatchNormalization(),
    Dropout(0.30),

    LSTM(
        64,
        return_sequences=False,
        activation="tanh"
    ),

    Dropout(0.30),

    Dense(
        128,
        activation="relu"
    ),

    Dropout(0.30),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

model.summary()

d:\ISL_Recognition_Project\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 128)        │       196,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 30, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 30, 128)        │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 30, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 20)             │         1,300 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 396,500 (1.51 MB)

 Trainable params: 395,988 (1.51 MB)

 Non-trainable params: 512 (2.00 KB)

In [16]:
# ============================================================
# COMPILE MODEL
# ============================================================

model.compile(
    optimizer=Adam(
        learning_rate=0.0005
    ),
    loss="categorical_crossentropy",
    metrics=["categorical_accuracy"]
)

print("Model compiled successfully.")

Model compiled successfully.


In [17]:
# ============================================================
# CALLBACKS
# ============================================================

early_stopping = EarlyStopping(
    monitor="val_categorical_accuracy",
    patience=20,
    mode="max",
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=7,
    min_lr=1e-6,
    verbose=1
)

checkpoint = ModelCheckpoint(
    MODEL_PATH,
    monitor="val_categorical_accuracy",
    mode="max",
    save_best_only=True,
    verbose=1
)

print("Callbacks ready.")

Callbacks ready.


In [18]:
# ============================================================
# TRAIN MODEL
# ============================================================

print("\n========================================")
print("STARTING TRAINING")
print("========================================\n")

history = model.fit(
    X_train,
    y_train_cat,

    validation_data=(
        X_val,
        y_val_cat
    ),

    epochs=150,

    batch_size=16,

    callbacks=[
        early_stopping,
        reduce_lr,
        checkpoint
    ],

    verbose=1
)

print("\nTraining finished.")


STARTING TRAINING

Epoch 1/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - categorical_accuracy: 0.0688 - loss: 2.9481 
Epoch 1: val_categorical_accuracy improved from None to 0.30000, saving model to D:\ISL_Recognition_Project\models\sign_model.keras

Epoch 1: finished saving model to D:\ISL_Recognition_Project\models\sign_model.keras
10/10 ━━━━━━━━━━━━━━━━━━━━ 12s 215ms/step - categorical_accuracy: 0.0688 - loss: 2.9481 - val_categorical_accuracy: 0.3000 - val_loss: 2.8849 - learning_rate: 5.0000e-04
Epoch 2/150
 9/10 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - categorical_accuracy: 0.1528 - loss: 2.8204
Epoch 2: val_categorical_accuracy improved from 0.30000 to 0.37500, saving model to D:\ISL_Recognition_Project\models\sign_model.keras

Epoch 2: finished saving model to D:\ISL_Recognition_Project\models\sign_model.keras
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 76ms/step - categorical_accuracy: 0.1688 - loss: 2.7993 - val_categorical_accuracy: 0.3750 - val_loss: 2.7700 - learning_rate: 5.0000e-04
Epoch 

In [19]:
# ============================================================
# LOAD BEST MODEL
# ============================================================

print("Loading best model...")

best_model = tf.keras.models.load_model(
    MODEL_PATH
)

print("Best model loaded successfully.")

Loading best model...
Best model loaded successfully.


In [20]:
# ============================================================
# FINAL VALIDATION
# ============================================================

loss, accuracy = best_model.evaluate(
    X_val,
    y_val_cat,
    verbose=0
)

print("\n========================================")
print("FINAL RESULT")
print("========================================")

print(f"Validation accuracy : {accuracy * 100:.2f}%")
print(f"Validation loss     : {loss:.4f}")


FINAL RESULT
Validation accuracy : 97.50%
Validation loss     : 0.1560


In [21]:
# ============================================================
# FILES SAVED
# ============================================================

print("\n========================================")
print("FILES SAVED")
print("========================================")

print("Model:")
print(MODEL_PATH)

print("\nNormalization:")
print(NORMALIZATION_PATH)

print("\nTraining completed successfully!")


FILES SAVED
Model:
D:\ISL_Recognition_Project\models\sign_model.keras

Normalization:
D:\ISL_Recognition_Project\models\normalization.npz

Training completed successfully!
